<a href="https://colab.research.google.com/github/priyanshisharma919346-del/Priya/blob/main/RAG%20AI%20Assistant%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader1 = PyPDFLoader("/content/sample_data/Data/WB_GBIOD.pdf")
loader2 = PyPDFLoader("/content/sample_data/Data/Clinmate change data.pdf")
loader3 = PyPDFLoader("/content/sample_data/Data/Soil data.pdf")
loader4 = PyPDFLoader("/content/sample_data/Data/WB_GBIOD.pdf")

# Combine all documents into one list
document = loader1.load() + loader2.load() + loader3.load() + loader4.load()

print(f"Total pages loaded: {len(document)}")

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader

def load_all_pdf(folder_path="/content/sample_data/Data"):
  num_doc = 0
  all_doc = []
  if os.path.exists(folder_path):
    for filename in os.listdir(folder_path):
      if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(folder_path, filename)
        loader = PyPDFLoader(pdf_path)
        doc = loader.load()
        all_doc.extend(doc)
        num_doc += 1
    print(f"Total PDFs loaded: {num_doc}")
    print(f"Total Pages loaded: {len(all_doc)}")
  else:
    print(f"Directory {folder_path} not found!")
  return all_doc


# PDFs load karein
load_pdf = load_all_pdf()

In [ ]:
# 1. Install & Import
!pip install langchain_text_splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Updated Splitter (Optimized Chunk Size & Overlap)
def split_chunk(documents, chunk_size=1000, chunk_overlap=150):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    chunk_doc = text_splitter.split_documents(documents)
    return chunk_doc

# 3. Chunks generate karein
chunks = split_chunk(load_pdf)
print(f"Total Chunks Created: {len(chunks)}")

In [ ]:
import os
import uuid
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer

# 1. Embedding Manager Class
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("Loading embedding model...", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("Embedding dimension:", self.model.get_sentence_embedding_dimension())

    def generate_embedding(self, text):
        # show_progress_bar=True helps track execution during batch encoding
        embeddings = self.model.encode(text, show_progress_bar=True)
        return embeddings


# 2. Vector Store Manager Class (ChromaDB)
class VectorStoreManager:
    def __init__(self, persist_directory="/content/sample_data/data/Vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None
        self.initialize_store()

    def initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name
        )
        print("Initialization of vector store collection:", self.collection_name)
        print("Existing documents in collection:", self.collection.count())

    def add_document(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents does not match number of embeddings")

        ids = []
        documents_content = []
        embedding_list = []
        all_metadata = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            # Clean and filter metadata for ChromaDB compatibility
            clean_metadata = {}
            if hasattr(doc, 'metadata') and isinstance(doc.metadata, dict):
                for k, v in doc.metadata.items():
                    if isinstance(v, (str, int, float, bool)):
                        clean_metadata[k] = v
                    else:
                        clean_metadata[k] = str(v)  # Convert complex structures to string

            clean_metadata["doc_index"] = i
            clean_metadata["content_length"] = len(doc.page_content)
            all_metadata.append(clean_metadata)

            documents_content.append(doc.page_content)

            # Safe numpy array to python list conversion
            if isinstance(emb, np.ndarray):
                embedding_list.append(emb.tolist())
            else:
                embedding_list.append(list(emb))

        # Batch Insertion to avoid memory timeout issues
        batch_size = 100
        for b in range(0, len(ids), batch_size):
            self.collection.add(
                ids=ids[b:b + batch_size],
                documents=documents_content[b:b + batch_size],
                embeddings=embedding_list[b:b + batch_size],
                metadatas=all_metadata[b:b + batch_size]
            )

        print("Total documents successfully added in vector storage:", len(documents_content))


# 3. Execution Setup
embedding_manager = EmbeddingManager()
vector_store = VectorStoreManager()

# Ensure 'chunks' is defined beforehand in your environment
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embedding(texts)

# Add data to ChromaDB
vector_store.add_document(chunks, embeddings)

In [ ]:
import os
from google.colab import userdata

GROQ_API_KEY = userdata.get('Key')
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("✅ Key connected successfully!")

In [ ]:
import os
import uuid
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq

import os
from google.colab import userdata

GROQ_API_KEY = userdata.get('key')
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# 2. PDF Loader
def load_all_pdf(folder_path="/content/sample_data/Data"):
    all_doc = []
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(".pdf"):
                pdf_path = os.path.join(folder_path, filename)
                loader = PyPDFLoader(pdf_path)
                all_doc.extend(loader.load())
    return all_doc

print("Loading PDFs...")
load_pdf = load_all_pdf()

# 3. Text Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(load_pdf)
print(f"Total chunks created: {len(chunks)}")

# 4. Classes Definition
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def generate_embedding(self, texts):
        return self.model.encode(texts, show_progress_bar=False)

class VectorStoreManager:
    def __init__(self, persist_directory="/content/sample_data/data/Vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_or_create_collection(name=self.collection_name)

    def add_document(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents does not match number of embeddings")

        ids, documents_content, embedding_list, all_metadata = [], [], [], []
        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            ids.append(f"doc_{uuid.uuid4()}")
            documents_content.append(doc.page_content)

            clean_metadata = {}
            if hasattr(doc, 'metadata') and isinstance(doc.metadata, dict):
                for k, v in doc.metadata.items():
                    if isinstance(v, (str, int, float, bool)):
                        clean_metadata[k] = v
                    else:
                        clean_metadata[k] = str(v)  # Convert complex structures to string
            clean_metadata["doc_index"] = i
            clean_metadata["content_length"] = len(doc.page_content)
            all_metadata.append(clean_metadata)

            embedding_list.append(emb.tolist() if isinstance(emb, np.ndarray) else list(emb))

        # Implement batching for ChromaDB to avoid exceeding max batch size
        batch_size = 500  # A reasonable batch size, adjust if needed
        for i in range(0, len(ids), batch_size):
            self.collection.add(
                ids=ids[i : i + batch_size],
                documents=documents_content[i : i + batch_size],
                embeddings=embedding_list[i : i + batch_size],
                metadatas=all_metadata[i : i + batch_size]
            )
        print("Successfully added documents. Total count in store:", self.collection.count())

# 5. Initialization & Populate Storage
embedding_manager = EmbeddingManager()
vector_store = VectorStoreManager()

# Agar collection khali hai, toh data add karein
if vector_store.collection.count() == 0 and chunks:
    print("Vector Store is empty. Adding document chunks now...")
    texts = [doc.page_content for doc in chunks]
    embeddings = embedding_manager.generate_embedding(texts)
    vector_store.add_document(chunks, embeddings)
else:
    print(f"Vector Store already contains {vector_store.collection.count()} documents.")

# 6. Rag Retrieval Setup
class Ragretrival:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrival(self, query, top_k=2):
        query_embedding = self.embedding_manager.generate_embedding([query])[0]
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )
        retrival_doc = []
        if results["documents"] and results["documents"][0]:
            for i, (doc_id, metadata, document, distance) in enumerate(zip(results["ids"][0], results["metadatas"][0], results["documents"][0], results["distances"][0])):
                # Use 1.0 / (1.0 + distance) for similarity score
                retrival_doc.append({"document": document, "similarity_score": 1.0 / (1.0 + distance)})
            print(f"Retrieved {len(retrival_doc)} documents")
        return retrival_doc

# Initialize rag_retrival and llm AFTER their classes are defined and vector store is populated
rag_retrival = Ragretrival(embedding_manager, vector_store)
llm = ChatGroq(model="groq/compound", groq_api_key=GROQ_API_KEY, temperature=0.2)

# 7. Output Generator
def generate_output(query, retriever, llm):
    results = retriever.retrival(query, top_k=2)
    context = "\n\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        return "No relevant context found in vector store."

    prompt = f"""You are an AI Environmental Scientist.
Analyze context across Soil Health, Water Availability, Metrics.
Context:
{context[:3500]}

Query:
{query}

STRICT Output Format:
1. What to do: [Actionable recommendation]
2. Impacted Metrics & Time Horizon: [Metrics affected + Timeline]
3. Scientific Reasoning & Reference: [Citation]
4. Confidence Level: [High/Medium/Low]
"""
    response = llm.invoke([prompt])
    return response.content

# 8. Run Query
user_query = "Soil organic carbon is 0.3%, low rainfall, semi-arid region. What should I do?"
answer = generate_output(user_query, rag_retrival, llm)
print(answer)

In [ ]:
user_query = "Effect of soil organic carbon on water retention?"

answer = generate_output(user_query, rag_retrival, llm)
print(answer)

In [ ]:
user_query = "What are the key soil management practices mentioned in the document?"
answer2 = generate_output(user_query, rag_retrival, llm)
print(answer2)

In [ ]:
# 1. Broad Summary Query
user_query = "Provide a comprehensive summary of all the main topics, findings, and key recommendations covered in the entire dataset/documents."

# Redefine generate_output to accept top_k argument for this cell's execution
def generate_output(query, retriever, llm, top_k=2):
    results = retriever.retrival(query, top_k=top_k)
    context = "\n\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        return "No relevant context found in vector store."

    prompt = f"""You are an AI Environmental Scientist.\nAnalyze context across Soil Health, Water Availability, Metrics.\nContext:\n{context[:3500]}\n\nQuery:\n{query}\n\nSTRICT Output Format:\n1. What to do: [Actionable recommendation]\n2. Impacted Metrics & Time Horizon: [Metrics affected + Timeline]\n3. Scientific Reasoning & Reference: [Citation]\n4. Confidence Level: [High/Medium/Low]\n"""
    response = llm.invoke([prompt])
    return response.content

# 2. Higher top_k
answer4 = generate_output(user_query, rag_retrival, llm, top_k=5)
print(answer4)

In [ ]:
user_query = "What specific practices are recommended to increase Soil Organic Carbon (SOC)?"

answer = generate_output(user_query, rag_retrival, llm)
print(answer)

In [ ]:
user_query = "What water conservation techniques are mentioned for semi-arid regions?"

# Direct output generate karein
answer = generate_output(user_query, rag_retrival, llm)
print(answer)